# Week 2 — From Predictions to a Full-Market Portfolio

A positive prediction for one stock is an entry signal. A portfolio model repeats that decision across **every stock in the market**, ranks the forecasts, and converts the best signals into weights.

This notebook trains matched MLP and LSTM predictors on every available EGX stock, rebalances weekly into the five strongest positive forecasts, charges 0.5% commission, and compares both portfolios with the real EGX30 benchmark on the same future period.

In [ ]:
import os, sys, json
while not os.path.isdir('src') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')
sys.path.insert(0, 'src')

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from pathlib import Path

from tradinglab.backtester import run_backtest
from tradinglab.data_feed import DataFeed
from tradinglab.features import FEATURE_NAMES, build_pooled_dataset, build_pooled_sequences
from tradinglab.ml import predict, train_model
from tradinglab.models import DeepMLP, LSTMRegressor
from tradinglab.simulator import PortfolioSimulator
from tradinglab.strategies.predictor import predictions_to_weights

SEED = 7
EPOCHS = 150
SEQ_LEN = 10
TOP_K = 5
REBALANCE_EVERY = 5
COMMISSION = 0.005
STARTING_CAPITAL = 1_000.0

torch.set_num_threads(1)
np.random.seed(SEED); torch.manual_seed(SEED)
plt.style.use('seaborn-v0_8-darkgrid')

## 1. Pool every stock without mixing past and future

The first 70% of calendar dates train both models. The final 30% is the common backtest period. Feature normalization is learned from training rows only and reused unchanged everywhere else.

In [ ]:
feed = DataFeed.from_dir('data/egx')
split_day = int(feed.n_days * 0.70)
X_train, y_train, X_test, y_test = build_pooled_dataset(feed, split_day)
Xtr_seq, ytr_seq, Xte_seq, yte_seq = build_pooled_sequences(feed, split_day, seq_len=SEQ_LEN)

x_mean = X_train.mean(axis=0)
x_std = X_train.std(axis=0)
x_std[x_std < 1e-8] = 1.0
X_train_scaled = ((X_train - x_mean) / x_std).astype('float32')
X_test_scaled = ((X_test - x_mean) / x_std).astype('float32')
Xtr_seq_scaled = ((Xtr_seq - x_mean) / x_std).astype('float32')
Xte_seq_scaled = ((Xte_seq - x_mean) / x_std).astype('float32')

print(f'Universe: {feed.n_assets} stocks')
print(feed.symbols)
print(f'Calendar: {feed.dates[0].date()} to {feed.dates[-1].date()}')
print(f'Test/backtest begins: {feed.dates[split_day].date()}')
print(f'MLP rows: {len(X_train_scaled):,} train / {len(X_test_scaled):,} test')
print(f'LSTM sequences: {len(Xtr_seq_scaled):,} train / {len(Xte_seq_scaled):,} test')
print('Features:', FEATURE_NAMES)

## 2. Train matched market-wide predictors

The MLP sees the latest feature row. The LSTM sees the latest 10 feature rows in order. Both learn from pooled examples across the entire universe and use the same seed, hidden width, epochs, and learning rate.

In [ ]:
torch.manual_seed(SEED)
mlp = DeepMLP(X_train_scaled.shape[1], hidden=32, n_hidden_layers=2)
mlp_history = train_model(
    mlp, X_train_scaled, y_train, X_test_scaled, y_test, epochs=EPOCHS, lr=1e-3
)

torch.manual_seed(SEED)
lstm = LSTMRegressor(Xtr_seq_scaled.shape[2], hidden=32)
lstm_history = train_model(
    lstm, Xtr_seq_scaled, ytr_seq, Xte_seq_scaled, yte_seq, epochs=EPOCHS, lr=1e-3
)

print(f'MLP parameters:  {sum(p.numel() for p in mlp.parameters()):,}')
print(f'LSTM parameters: {sum(p.numel() for p in lstm.parameters()):,}')
print(f'MLP final test MSE:  {mlp_history["test"][-1]:.8f}')
print(f'LSTM final test MSE: {lstm_history["test"][-1]:.8f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
axes[0].plot(mlp_history['train'], label='Train')
axes[0].plot(mlp_history['test'], label='Test')
axes[0].set(title='MLP loss', xlabel='Epoch', ylabel='MSE')
axes[0].legend()
axes[1].plot(lstm_history['train'], label='Train')
axes[1].plot(lstm_history['test'], label='Test')
axes[1].set(title='LSTM loss', xlabel='Epoch')
axes[1].legend()
plt.tight_layout(); plt.show()

## 3. Predictions → ranking → equal-weight portfolio

At a rebalance, the model scores every stock in `feed.symbols`, keeps the five highest **positive** predictions, and assigns each selected stock 20%. If every prediction is negative, it holds cash. Between weekly rebalances, weights remain unchanged.

In [ ]:
def periodic_model_strategy(model, architecture, top_k=5, rebalance_every=5):
    state = {
        'calls': 0,
        'weights': np.zeros(feed.n_assets),
        'predictions': np.full(feed.n_assets, np.nan),
    }

    def strategy(observation):
        if state['calls'] % rebalance_every == 0:
            if architecture == 'mlp':
                model_input = (observation[:, -1, :] - x_mean) / x_std
            else:
                model_input = (observation[:, -SEQ_LEN:, :] - x_mean) / x_std
            state['predictions'] = predict(model, model_input.astype('float32'))
            state['weights'] = predictions_to_weights(state['predictions'], top_k=top_k)
        state['calls'] += 1
        return state['weights'].copy()

    return strategy, state

mlp_strategy, mlp_state = periodic_model_strategy(mlp, 'mlp', TOP_K, REBALANCE_EVERY)
lstm_strategy, lstm_state = periodic_model_strategy(lstm, 'lstm', TOP_K, REBALANCE_EVERY)

simulator = PortfolioSimulator(feed, benchmark='egx30', commission=COMMISSION)
result_mlp = run_backtest(simulator, mlp_strategy, lookback=30, start=split_day)
result_lstm = run_backtest(simulator, lstm_strategy, lookback=30, start=split_day)

print(f'Backtest sessions: {len(result_mlp["dates"]):,}')
print(f'Rebalance frequency: every {REBALANCE_EVERY} sessions')
print(f'Commission: {COMMISSION:.1%} per unit of turnover')

In [ ]:
def portfolio_metrics(result):
    returns = result['portfolio_returns']
    equity = result['portfolio']
    curve = np.r_[1.0, equity]
    drawdown = curve / np.maximum.accumulate(curve) - 1.0
    volatility = returns.std(ddof=1)
    sharpe = np.sqrt(252) * returns.mean() / volatility if volatility > 0 else np.nan
    weights = result['weights']
    previous = np.vstack([np.zeros(feed.n_assets), weights[:-1]])
    turnover = np.abs(weights - previous).sum(axis=1) / 2.0
    previous_equity = np.r_[STARTING_CAPITAL, equity[:-1] * STARTING_CAPITAL]
    fees_paid = float(np.sum(previous_equity * COMMISSION * turnover))
    return {
        'Ending EGP': equity[-1] * STARTING_CAPITAL,
        'Total return': equity[-1] - 1.0,
        'Sharpe': sharpe,
        'Max drawdown': drawdown.min(),
        'Rebalances': int(np.count_nonzero(turnover > 1e-12)),
        'Total turnover': turnover.sum(),
        'Fees paid': fees_paid,
    }

benchmark_equity = result_mlp['benchmark']
benchmark_returns = result_mlp['benchmark_returns']
benchmark_curve = np.r_[1.0, benchmark_equity]
benchmark_drawdown = benchmark_curve / np.maximum.accumulate(benchmark_curve) - 1.0
benchmark_volatility = benchmark_returns.std(ddof=1)
benchmark_sharpe = (
    np.sqrt(252) * benchmark_returns.mean() / benchmark_volatility
    if benchmark_volatility > 0 else np.nan
)
results = pd.DataFrame({
    'MLP portfolio': portfolio_metrics(result_mlp),
    'LSTM portfolio': portfolio_metrics(result_lstm),
}).T
results.loc['EGX30 benchmark'] = {
    'Ending EGP': benchmark_equity[-1] * STARTING_CAPITAL,
    'Total return': benchmark_equity[-1] - 1.0,
    'Sharpe': benchmark_sharpe,
    'Max drawdown': benchmark_drawdown.min(),
    'Rebalances': 0,
    'Total turnover': 0.0,
    'Fees paid': 0.0,
}

fig, ax = plt.subplots(figsize=(13, 5.5))
ax.plot(result_mlp['dates'], result_mlp['portfolio'] * STARTING_CAPITAL, label='MLP portfolio')
ax.plot(result_lstm['dates'], result_lstm['portfolio'] * STARTING_CAPITAL, label='LSTM portfolio')
ax.plot(result_mlp['dates'], benchmark_equity * STARTING_CAPITAL, label='EGX30 benchmark', linestyle='--')
ax.set(title='Full-market neural portfolios after commission', xlabel='Date', ylabel='Portfolio value (EGP)')
ax.legend(); plt.tight_layout(); plt.show()

display(results.style.format({
    'Ending EGP': '{:,.2f}', 'Total return': '{:.2%}', 'Sharpe': '{:.3f}',
    'Max drawdown': '{:.2%}', 'Rebalances': '{:.0f}', 'Total turnover': '{:.1f}',
    'Fees paid': '{:,.2f}',
}))

winner = results.loc[['MLP portfolio', 'LSTM portfolio'], 'Ending EGP'].idxmax()
benchmark_end = benchmark_equity[-1] * STARTING_CAPITAL
print(f'MODEL WINNER: {winner}')
print(f'Beat EGX30: {"yes" if results.loc[winner, "Ending EGP"] > benchmark_end else "no"}')

## 4. Make the model's portfolio decision visible

The tables below show the final rebalance decision: every stock was scored, but only the highest positive predictions received portfolio weight.

In [ ]:
def latest_decision(state, name):
    table = pd.DataFrame({
        'Symbol': feed.symbols,
        'Predicted next-day return': state['predictions'],
        'Portfolio weight': state['weights'],
    }).sort_values('Predicted next-day return', ascending=False)
    print(name)
    display(table.head(10).style.format({
        'Predicted next-day return': '{:+.5f}', 'Portfolio weight': '{:.0%}'
    }))

latest_decision(mlp_state, 'MLP — highest forecasts at final rebalance')
latest_decision(lstm_state, 'LSTM — highest forecasts at final rebalance')

In [ ]:
def result_metrics_payload(row_name):
    row = results.loc[row_name]
    return {
        'final_equity': round(float(row['Ending EGP']), 2),
        'total_return': round(float(row['Total return']), 6),
        'sharpe': round(float(row['Sharpe']), 6),
        'max_drawdown': round(float(row['Max drawdown']), 6),
        'rebalances': int(row['Rebalances']),
        'turnover': round(float(row['Total turnover']), 4),
        'fees_paid': round(float(row['Fees paid']), 2),
    }

def holdings_payload(state):
    rows = []
    for symbol, prediction, weight in zip(feed.symbols, state['predictions'], state['weights']):
        if weight > 0:
            rows.append({
                'symbol': symbol,
                'prediction': round(float(prediction), 8),
                'weight': round(float(weight), 6),
            })
    return sorted(rows, key=lambda row: row['prediction'], reverse=True)

dashboard_payload = {
    'dates': result_mlp['dates'].strftime('%Y-%m-%d').tolist(),
    'mlp': (result_mlp['portfolio'] * STARTING_CAPITAL).round(4).tolist(),
    'lstm': (result_lstm['portfolio'] * STARTING_CAPITAL).round(4).tolist(),
    'benchmark': (benchmark_equity * STARTING_CAPITAL).round(4).tolist(),
    'symbols': feed.symbols,
    'settings': {
        'top_k': TOP_K,
        'rebalance_every': REBALANCE_EVERY,
        'commission': COMMISSION,
        'starting_capital': STARTING_CAPITAL,
        'sequence_length': SEQ_LEN,
        'benchmark': 'EGX30',
    },
    'metrics': {
        'mlp': result_metrics_payload('MLP portfolio'),
        'lstm': result_metrics_payload('LSTM portfolio'),
        'benchmark': result_metrics_payload('EGX30 benchmark'),
    },
    'holdings': {
        'mlp': holdings_payload(mlp_state),
        'lstm': holdings_payload(lstm_state),
    },
}
output_path = Path('dashboard/data/neural_portfolio.json')
output_path.write_text(json.dumps(dashboard_payload, indent=2), encoding='utf-8')
print(f'Saved dashboard data to {output_path}')

## Takeaway

The neural network does not directly choose money amounts. It predicts one return per stock; ranking converts those forecasts into a long-only portfolio; the simulator applies the next session's real returns and subtracts turnover costs. Prediction quality, ranking quality, diversification, rebalance frequency, and commission all determine the final portfolio result.